# OpenWeather air-pollution comparison data

This historical notebook downloads OpenWeather air-pollution records for sampled rice locations or district centroids and constructs monthly PM2.5 comparison data.
The paper's main PM2.5 extraction uses CAMS in [`replication/extract_covariates.py`](replication/extract_covariates.py).
OpenWeather is a separate comparison source, not a substitute for that extraction.

## Setup and inputs

Start Jupyter from the repository root.
The notebook uses `requests`, `pandas`, `geopandas`, `shapely`, and `tqdm`.
Provide `datasets/South_sample_5x10000.csv` with `sample_4`, `lat`, and `lon`; the district branch additionally uses `JAXA_subtiles.json` and `gadm41_VNM_2.json`.
The historical root-level tile filenames differ from the current `shapefiles/` layout, so point them to the intended archived inputs.
Create `datasets/owm/` and `owm_pollution/` before running the corresponding download loop.
Set `OPENWEATHER_API_KEY` in the kernel's environment before making requests; no credential is stored in this notebook.

## Reading and execution

Sections 1 and 3 form the sampled-location workflow for 2022.
Section 2 is a separate district extraction experiment for 2018-2022; its files are not consumed by the sample aggregation below.
The download cells make external requests and overwrite their named JSON outputs when executed.
The saved debug output and execution counts have been cleared; no download or analysis was rerun during annotation.
For structural and syntax checks, see [`NOTEBOOKS.md`](NOTEBOOKS.md).

Historical date conversion uses the machine's local timezone through `time.mktime`, and endpoints are midnight on the named last day.
The district centroids are calculated directly in geographic coordinates, and the district loop resumes at index 290.
These choices are retained and called out where used; this notebook does not establish a complete fresh district run.


## 1. Sampled rice locations

### Load dependencies and the external API key

Only libraries used by the retained code are imported.
The key is needed only by the download functions; existing JSON files can be aggregated without making requests.

In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from tqdm.notebook import tqdm

dir = os.getcwd()
api_key = os.environ.get("OPENWEATHER_API_KEY", "")

### Load the supplied sampling table

The active input is the five-by-10,000 sampling table.
The older three-percent input remains a commented historical alternative.

In [ ]:
# df = pd.read_csv(os.path.join(dir, 'datasets/South_VNM_3per_sampleID.csv'))
df = pd.read_csv(os.path.join(dir, 'datasets/South_sample_5x10000.csv'))
df

### Select the fourth sample

The exact filter is `sample_4 == 1`.
No random sample is drawn in this cell.

In [ ]:
# Filter by sample
sample = df[df['sample_4'] == 1]
sample

### Download the 2022 record for each selected coordinate

The request loop uses January 1 through December 31 as locally converted Unix timestamps.
It writes one JSON per latitude/longitude pair to `datasets/owm/`.
The code retains the original request and response handling; inspect API response errors before aggregation.
This is the first of two historical `create_json` definitions; the district section replaces it.

In [ ]:
lats = sample['lat'].to_list()
lons = sample['lon'].to_list()

def create_json(lat:str, lon:str, start: int, end: int) -> dict:
    """Makes a request to the OpenWeatherMap API to get air pollution data for a given location and time period."""
    if not api_key:
        raise ValueError("Set OPENWEATHER_API_KEY before requesting data")
    hist_url = f'https://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start}&end={end}&appid={api_key}'
    r = requests.get(hist_url)
    json_out = r.json()

    return json_out

# Loop through the coordinates and create a json file for each pair
for i in tqdm(range(len(lats))[:]):

    print(lats[i], lons[i])


    start_1 = int(time.mktime(datetime(2022, 1, 1).timetuple()))
    end_1 = int(time.mktime(datetime(2022, 12, 31).timetuple()))

    # Dumps json file
    json_1 = create_json(lats[i], lons[i], start_1, end_1)
    with open(f"datasets/owm/{lats[i]}_{lons[i]}_2022.json", "w") as outfile:
        json.dump(json_1, outfile)

## 2. Separate district-centroid extraction

### Select the historical tile slices

Selection depends on JSON insertion order: indices 1-15 and 33 onward are included.
The resulting WGS84 polygons are written to `chosen_tiles.json`.
These slices should not be read as a documented full study-area selection.

In [ ]:
# JAXA LULC tiles
with open('JAXA_subtiles.json', 'r') as json_file:
    jaxa_subtiles = json.load(json_file)

tiles = list(jaxa_subtiles.keys())[1:16]
tiles.extend(list(jaxa_subtiles.keys())[33:])

# Bounding box of the chosen tiles
bbox = [jaxa_subtiles[tile] for tile in tiles]
bbox = [box(tile[0], tile[1], tile[2], tile[3]) for tile in bbox]

# Create a GeoDataFrame from the tiles and their bounding boxes
tiles_gdf = gpd.GeoDataFrame({'tile': tiles,
                              'geometry': bbox})
tiles_gdf = tiles_gdf.set_crs(epsg=4326)
tiles_gdf.head()
tiles_gdf.to_file('chosen_tiles.json', driver='GeoJSON', index=False)

### Load GADM district boundaries

The district branch uses GADM 4.1 level 2 boundaries in the same coordinate system as the selected tiles.

In [ ]:
# Loads VNM's districts
dist_gdf = gpd.read_file('gadm41_VNM_2.json')
dist_gdf = dist_gdf.to_crs(tiles_gdf.crs)
dist_gdf.head()

### Keep intersecting districts and derive centroid coordinates

A district is retained if it intersects any selected tile.
Centroids are calculated in EPSG:4326 rather than a projected CRS, preserving the original method.
The set conversion and source feature ordering affect the later resume slice.

In [ ]:
# Check which districts intersect with the chosen tiles
dist_geom = [geom for geom in dist_gdf.geometry]

intersects_geom = []
for geom in dist_geom:
    gdf = tiles_gdf.iloc[tiles_gdf.sindex.query(geom, predicate='intersects')]
    if len(gdf) > 0:
        intersects_geom.append(geom)

intersects_geom = list(set(intersects_geom))
dist_filtered = dist_gdf[dist_gdf.geometry.isin(intersects_geom)]

# Get the centroid of the districts
dist_filtered['lon'] = dist_filtered.centroid.x
dist_filtered['lat'] = dist_filtered.centroid.y
dist_filtered = dist_filtered[['GID_2', 'lat', 'lon']]
dist_dict = dist_filtered.set_index('GID_2').to_dict('index')

### Resume the district downloads

This cell redefines `create_json` for district IDs and starts at district index 290.
It requests two date windows per year from 2018 through 2022 and writes the `HN_...` filenames used by the experiment.
The half-year endpoints are June 30 and December 31 at local midnight; they are not automatically extended to include each final day.

In [ ]:
def create_json(location_name:str, start: int, end: int) -> dict:
    """Makes a request to the OpenWeatherMap API to get air pollution data for a given district and time period."""
    global dist_dict
    lat = dist_dict[location_name]['lat']
    lon = dist_dict[location_name]['lon']
    if not api_key:
        raise ValueError("Set OPENWEATHER_API_KEY before requesting data")
    hist_url = f'https://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start}&end={end}&appid={api_key}'
    r = requests.get(hist_url)
    json_out = r.json()
    json_out['name'] = location_name

    return json_out

for location_name in list(dist_dict.keys())[290:]:

    print(location_name)

    # 6 month extraction to avoid request timeout from OWM

    for year in list(range(2018, 2023, 1)):
        print('     ', year)
        # January - June
        start_1 = int(time.mktime(datetime(year, 1, 1).timetuple()))
        end_1 = int(time.mktime(datetime(year, 6, 30).timetuple()))

        # Dumps json file
        json_1 = create_json(location_name, start_1, end_1)
        with open(f"owm_pollution/HN_{location_name}_16_{year}.json", "w") as outfile:
            json.dump(json_1, outfile)

        # July - December
        start_2 = int(time.mktime(datetime(year, 7, 1).timetuple()))
        end_2 = int(time.mktime(datetime(year, 12, 31).timetuple()))

        # Dumps json file
        json_2 = create_json(location_name, start_2, end_2)
        with open(f"owm_pollution/HN_{location_name}_712_{year}.json", "w") as outfile:
            json.dump(json_2, outfile)

## 3. Aggregate the sampled-location JSON files

The input folder is `datasets/owm/`, not the separate district output folder.
Expected basenames are `latitude_longitude_2022.json`.
The filename parser uses the basename so underscores in parent folders do not affect coordinates.
API timestamps are interpreted as UTC, converted to `Asia/Saigon`, and averaged by calendar month.
Aggregation runs separately within each input file, and grouping omits year because this branch assumes 2022 data.
A file spanning years combines like-numbered months; separate yearly files instead produce repeated location/month keys after concatenation.
The final merge joins the original sampling table on exact numeric latitude and longitude.

In [ ]:
# All the json files in the directory
paths = sorted(Path('datasets/owm').glob('*.json'))
paths.sort()
dfs = []

# Loop through the json files and extract the data
# Creates a dataframe for each json file
# Concatenates the list into a single dataframe
for path in tqdm(paths):
    with open(path, 'r') as f:
        d = json.load(f)
    dates = []
    pm25s = []

    for i in range(len(d['list'])):
        dates.append(d['list'][i]['dt'])
        pm25s.append(d['list'][i]['components']['pm2_5'])


    latitude, longitude, year_label = path.stem.rsplit('_', 2)

    df_p = pd.DataFrame({'lat': latitude,
                        'lon': longitude,
                        'dt': dates,
                        'pm25': pm25s})
    df_p['dt'] = pd.to_datetime(df_p['dt'], unit='s')
    df_p['dt'] = df_p['dt'].dt.tz_localize('UTC').dt.tz_convert('Asia/Saigon')
    df_p['month'] = df_p['dt'].dt.month
    df_p = df_p.groupby(['lat', 'lon', 'month']).agg({'pm25': 'mean'}).reset_index()

    df_p['lat'] = df_p['lat'].astype('float64')
    df_p['lon'] = df_p['lon'].astype('float64')

    dfs.append(df_p)

df_all = pd.concat(dfs, axis=0).reset_index(drop=True)

# Merges the sample data with the pollution data
df_out = pd.merge(df, df_all, on=['lat', 'lon'], how='inner')

### Write the monthly sample comparison table

This writes `datasets/owm_pm25_2022_2nd_batch.csv`.
The name is preserved from the historical notebook, despite the active `sample_4` selection above.
Do not infer the sample identifier from the output filename alone.

In [ ]:
# Saves the output to a csv file
df_out.to_csv('datasets/owm_pm25_2022_2nd_batch.csv', index=False)